# 📊 Data Analyst Job Market — India

An analysis of the data job market in India, focused on Data Analyst roles: skill demand, salary trends, and the intersection of the two, across the country's major tech hubs.

> **Data note:** This analysis runs on a **simulated dataset of 6,000 India-based postings**, generated to match publicly reported 2026 salary benchmarks (Glassdoor, PayScale, Indeed, GROWAI) for role- and city-level pay, and realistic skill-demand patterns across Indian job portals. It is not scraped live posting data. Every figure below is a direct computed output of the pandas/seaborn pipeline in this notebook — run all cells top to bottom to reproduce every number exactly.

**Repo:** https://github.com/Harshitha954/data-analyst-job-market-india

Run this notebook directly in Colab — just click **Runtime → Run all**. No setup needed.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import random
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime
import json

sns.set_theme(style="whitegrid", palette="dark:b_r")
np.random.seed(42)
random.seed(42)

## 2. Generate the (simulated) dataset

6,000 job postings calibrated against researched 2026 salary benchmarks for city- and role-level pay, and realistic skill-demand patterns.

In [ ]:
N_JOBS = 6000

CITIES = {
    "Bengaluru":  {"weight": 0.27, "salary_mult": 1.17},
    "Hyderabad":  {"weight": 0.18, "salary_mult": 1.08},
    "Pune":       {"weight": 0.13, "salary_mult": 1.04},
    "Delhi NCR":  {"weight": 0.14, "salary_mult": 1.05},
    "Mumbai":     {"weight": 0.13, "salary_mult": 1.10},
    "Chennai":    {"weight": 0.09, "salary_mult": 0.97},
    "Kolkata":    {"weight": 0.03, "salary_mult": 0.85},
    "Ahmedabad":  {"weight": 0.03, "salary_mult": 0.88},
}

ROLES = {
    "Data Analyst":   {"weight": 0.47, "base_lpa": (3.5, 11.5)},
    "Data Scientist": {"weight": 0.27, "base_lpa": (6.0, 23.0)},
    "Data Engineer":  {"weight": 0.26, "base_lpa": (5.5, 21.0)},
}

SKILLS = {
    "SQL":        ("Analyst Tool", 0.0),
    "Excel":      ("Analyst Tool", -0.8),
    "PowerPoint": ("Analyst Tool", -1.0),
    "Power BI":   ("Analyst Tool", 0.6),
    "Tableau":    ("Analyst Tool", 1.0),
    "Python":     ("Programming", 2.2),
    "R":          ("Programming", 1.6),
    "SAS":        ("Programming", 1.2),
    "Scala":      ("Programming", 3.0),
    "AWS":        ("Cloud", 2.8),
    "Azure":      ("Cloud", 2.6),
    "GCP":        ("Cloud", 2.7),
    "Spark":      ("Cloud", 2.9),
    "Snowflake":  ("Database", 3.1),
    "Oracle":     ("Database", 2.3),
    "SQL Server": ("Database", 1.4),
    "Kafka":      ("Cloud", 2.8),
    "Airflow":    ("Cloud", 2.5),
    "Git":        ("Programming", 0.9),
    "Docker":     ("Cloud", 2.0),
}

ROLE_SKILL_PROB = {
    "Data Analyst": {
        "SQL": 0.72, "Excel": 0.68, "PowerPoint": 0.34, "Power BI": 0.38,
        "Tableau": 0.33, "Python": 0.41, "R": 0.14, "SAS": 0.09,
        "SQL Server": 0.16, "Oracle": 0.08, "Git": 0.10,
    },
    "Data Scientist": {
        "Python": 0.81, "SQL": 0.62, "R": 0.29, "AWS": 0.27, "Azure": 0.20,
        "Spark": 0.24, "Tableau": 0.18, "SAS": 0.13, "Git": 0.22, "GCP": 0.15,
        "Docker": 0.17,
    },
    "Data Engineer": {
        "Python": 0.74, "SQL": 0.66, "AWS": 0.39, "Azure": 0.31, "Spark": 0.44,
        "Airflow": 0.28, "Kafka": 0.24, "Scala": 0.18, "Snowflake": 0.21,
        "GCP": 0.22, "Docker": 0.26, "Git": 0.19,
    },
}

MONTHS = list(range(1, 13))
EXCEL_SEASONALITY = {m: 1.0 + (0.35 * (m - 6) / 6 if m >= 9 else 0.0) for m in MONTHS}
POSTING_SEASONALITY = {1: 1.05, 2: 1.0, 3: 1.05, 4: 0.95, 5: 0.9, 6: 0.95,
                        7: 1.0, 8: 1.0, 9: 1.05, 10: 1.1, 11: 1.05, 12: 0.85}

rows = []
job_id = 100000

city_keys = list(CITIES.keys())
city_weights = [CITIES[c]["weight"] for c in city_keys]

role_keys = list(ROLES.keys())
role_weights = [ROLES[r]["weight"] for r in role_keys]

for _ in range(N_JOBS):
    job_id += 1
    role = random.choices(role_keys, weights=role_weights, k=1)[0]
    city = random.choices(city_keys, weights=city_weights, k=1)[0]
    month = random.choices(MONTHS, weights=[POSTING_SEASONALITY[m] for m in MONTHS], k=1)[0]
    day = random.randint(1, 28)
    posted_date = datetime(2025, month, day)

    exp_level = np.random.choice(
        ["fresher", "junior", "mid", "senior"], p=[0.28, 0.32, 0.26, 0.14]
    )
    lo, hi = ROLES[role]["base_lpa"]
    span = hi - lo
    exp_position = {
        "fresher": np.random.uniform(0.0, 0.20),
        "junior":  np.random.uniform(0.15, 0.45),
        "mid":     np.random.uniform(0.40, 0.70),
        "senior":  np.random.uniform(0.65, 1.0),
    }[exp_level]
    base_salary = lo + span * exp_position

    skill_probs = ROLE_SKILL_PROB[role]
    posting_skills = []
    for skill, prob in skill_probs.items():
        p = prob
        if role == "Data Analyst" and skill == "Excel":
            p = min(0.95, prob * EXCEL_SEASONALITY[month])
        if random.random() < p:
            posting_skills.append(skill)
    if not posting_skills:
        posting_skills = [max(skill_probs, key=skill_probs.get)]

    premium = sum(SKILLS[s][1] for s in posting_skills if s in SKILLS)
    city_mult = CITIES[city]["salary_mult"]
    noise = np.random.normal(0, 0.8)
    salary_lpa = max(3.0, (base_salary + premium) * city_mult + noise)

    rows.append({
        "job_id": job_id,
        "job_title_short": role,
        "job_city": city,
        "job_country": "India",
        "job_posted_date": posted_date,
        "experience_level": exp_level,
        "salary_year_lpa": round(salary_lpa, 2),
        "job_skills": posting_skills,
    })

df = pd.DataFrame(rows)
df["month"] = df["job_posted_date"].dt.month
print(df.shape)
df.head()

## 3. Explode skills into one row per (job, skill)

In [ ]:
exploded = df.explode("job_skills").rename(columns={"job_skills": "job_skill"})
results = {}

results["total_jobs"] = int(len(df))
results["da_jobs"] = int((df["job_title_short"] == "Data Analyst").sum())
results["ds_jobs"] = int((df["job_title_short"] == "Data Scientist").sum())
results["de_jobs"] = int((df["job_title_short"] == "Data Engineer").sum())
results["skills_analyzed"] = int(exploded["job_skill"].nunique())
results["cities_count"] = int(df["job_city"].nunique())
results["median_salary_overall"] = float(df["salary_year_lpa"].median())
results["median_salary_da"] = float(df.loc[df["job_title_short"] == "Data Analyst", "salary_year_lpa"].median())
print(json.dumps(results, indent=2))

## 4. Most in-demand skills by role

In [ ]:
job_titles = df["job_title_short"].value_counts().index.tolist()
role_totals = df["job_title_short"].value_counts()

skill_counts = exploded.groupby(["job_title_short", "job_skill"]).size().rename("skill_count").reset_index()
skill_counts["skill_percent"] = skill_counts.apply(
    lambda r: 100 * r["skill_count"] / role_totals[r["job_title_short"]], axis=1
)

fig, ax = plt.subplots(len(job_titles), 1, figsize=(8, 10))
for i, job_title in enumerate(job_titles):
    df_plot = (skill_counts[skill_counts["job_title_short"] == job_title]
               .sort_values("skill_percent", ascending=False).head(5)[::-1])
    sns.barplot(data=df_plot, x="skill_percent", y="job_skill", ax=ax[i], hue="skill_count",
                palette="dark:b_r", legend=False)
    ax[i].set_title(job_title, fontsize=12, fontweight="bold")
    ax[i].set_xlabel("% of Postings" if i == len(job_titles)-1 else "")
    ax[i].set_ylabel("")
    ax[i].xaxis.set_major_formatter(mticker.PercentFormatter())
plt.suptitle("Likelihood of Skills Requested in Indian Job Postings", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Skill trend for Data Analysts across 2025

In [ ]:
da_exploded = exploded[exploded["job_title_short"] == "Data Analyst"]
da_monthly_total = df[df["job_title_short"] == "Data Analyst"].groupby("month").size()

top5_da_skills = (da_exploded["job_skill"].value_counts().head(5).index.tolist())
trend = (da_exploded[da_exploded["job_skill"].isin(top5_da_skills)]
         .groupby(["month", "job_skill"]).size().unstack(fill_value=0))
trend_pct = trend.div(da_monthly_total, axis=0) * 100
trend_pct = trend_pct[top5_da_skills]

plt.figure(figsize=(9, 5))
sns.lineplot(data=trend_pct, dashes=False, palette="tab10")
plt.gca().yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
plt.title("Trending Top Skills for Data Analysts in India (2025)", fontsize=13, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("% of Postings")
plt.xticks(range(1, 13))
plt.legend(title="Skill", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 6. Salary distribution across roles

In [ ]:
job_order = df.groupby("job_title_short")["salary_year_lpa"].median().sort_values(ascending=False).index

plt.figure(figsize=(9, 4.5))
sns.boxplot(data=df, x="salary_year_lpa", y="job_title_short", order=job_order, hue="job_title_short",
            palette="dark:b_r", legend=False)
plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda y, pos: f"\u20b9{y:.0f}L"))
plt.title("Salary Distributions of Data Jobs in India", fontsize=13, fontweight="bold")
plt.xlabel("Median Annual Salary")
plt.ylabel("")
plt.tight_layout()
plt.show()

df.groupby("job_title_short")["salary_year_lpa"].median().round(2)

## 7. Highest-paid vs. most in-demand skills (Data Analyst)

In [ ]:
da_only_exploded = exploded[exploded["job_title_short"] == "Data Analyst"]
da_skill_stats = da_only_exploded.groupby("job_skill").agg(
    median_salary=("salary_year_lpa", "median"),
    count=("salary_year_lpa", "size")
)
da_skill_stats["skill_percent"] = 100 * da_skill_stats["count"] / results["da_jobs"]

top_pay = da_skill_stats.sort_values("median_salary", ascending=False).head(10)
top_demand = da_skill_stats.sort_values("count", ascending=False).head(10)

fig, ax = plt.subplots(2, 1, figsize=(8, 9))
sns.barplot(data=top_pay.reset_index(), x="median_salary", y="job_skill", hue="median_salary",
            ax=ax[0], palette="dark:b_r", legend=False)
ax[0].set_title("Top 10 Highest Paid Skills for Data Analysts (India)", fontweight="bold")
ax[0].set_xlabel("Median Salary (\u20b9 LPA)")
ax[0].set_ylabel("")

sns.barplot(data=top_demand.reset_index(), x="count", y="job_skill", hue="count",
            ax=ax[1], palette="light:b", legend=False)
ax[1].set_title("Top 10 Most In-Demand Skills for Data Analysts (India)", fontweight="bold")
ax[1].set_xlabel("Number of Postings")
ax[1].set_ylabel("")
plt.tight_layout()
plt.show()

## 8. Salary by city (Data Analyst)

In [ ]:
da_df = df[df["job_title_short"] == "Data Analyst"]
city_salary = da_df.groupby("job_city")["salary_year_lpa"].median().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=city_salary.values, y=city_salary.index, hue=city_salary.index,
            palette="mako", legend=False)
plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda y, pos: f"\u20b9{y:.0f}L"))
plt.title("Median Data Analyst Salary by Indian City", fontsize=13, fontweight="bold")
plt.xlabel("Median Annual Salary")
plt.ylabel("")
plt.tight_layout()
plt.show()

city_salary.round(2)

## 9. Most optimal skills for Data Analysts (demand vs. pay)

In [ ]:
skill_category = {
    "SQL": "Analyst Tool", "Excel": "Analyst Tool", "PowerPoint": "Analyst Tool",
    "Power BI": "Analyst Tool", "Tableau": "Analyst Tool", "Python": "Programming",
    "R": "Programming", "SAS": "Programming", "SQL Server": "Database", "Oracle": "Database",
    "Git": "Programming",
}

da_skill_stats_reset = da_skill_stats.reset_index()
da_skill_stats_reset["technology"] = da_skill_stats_reset["job_skill"].map(skill_category).fillna("Other")

high_demand = da_skill_stats_reset[da_skill_stats_reset["count"] >= da_skill_stats_reset["count"].quantile(0.25)]

plt.figure(figsize=(8, 6))
sns.scatterplot(data=high_demand, x="skill_percent", y="median_salary", hue="technology",
                 palette="bright", s=110, legend="full")
for _, row in high_demand.iterrows():
    plt.text(row["skill_percent"] + 0.4, row["median_salary"], row["job_skill"], fontsize=8)
plt.gca().xaxis.set_major_formatter(mticker.PercentFormatter())
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, pos: f"\u20b9{y:.0f}L"))
plt.title("Most Optimal Skills for Data Analysts in India", fontsize=13, fontweight="bold")
plt.xlabel("% of Data Analyst Postings")
plt.ylabel("Median Salary")
plt.tight_layout()
plt.show()

## Conclusion

Across 6,000 analyzed postings, Excel and SQL are the clear baseline for Data Analyst roles in India, appearing in 73.6% and 71.9% of postings respectively, while Python offers the best combination of demand (41.4%) and pay (₹9.68 LPA median) among all skills studied. Bengaluru leads both in posting volume and salary, and the gap between the highest- and lowest-paying skills for the same role exceeds ₹2 LPA — evidence that skill choice, not just job title, meaningfully shapes earning potential in India's data analytics market.

**Full write-up and static charts:** https://github.com/Harshitha954/data-analyst-job-market-india